# Social-Properties Synthetic Data Toy Pilot

This scratch notebook tests a small chained prompting pipeline for everyday social-property examples:

`social-property scenario plan -> prose rendering with discourse style -> lightweight edit/check -> programmatic audit -> final prose export`

The intended latent structure is:

`behavioral evidence -> social axis -> inferred social property`

The important safety constraint is that social/personality-like traits must be inferred from behavior in the scenario, not from identity, appearance, accent, race, ethnicity, religion, gender, nationality, disability, class markers, or family background.

This is an MVP for prompt quality, Qwen reliability, JSON parseability, prose diversity, and audit logic. It does not train a BabyLM model and it does not try to build the production pipeline.


## 1. Setup And Configuration

Set `QWEN_MODEL_ID` if you want a different local model path or backend model name. Set `SOCIAL_PROPERTIES_DRY_RUN=1` to test the notebook logic without loading a model.


In [ ]:
import copy
import json
import math
import os
import random
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path


# Keep this before importing torch/transformers/vLLM. Physical GPUs 4,5 become visible as cuda:0,cuda:1.
DEFAULT_VISIBLE_GPUS = os.environ.get("SOCIAL_PROPERTIES_CUDA_VISIBLE_DEVICES", "4,5")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
_already_loaded_cuda_libs = [name for name in ("torch", "transformers", "vllm") if name in sys.modules]
if _already_loaded_cuda_libs:
    print("WARNING: torch/transformers/vLLM were already imported; restart the kernel for CUDA_VISIBLE_DEVICES changes to take effect.")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", DEFAULT_VISIBLE_GPUS)

MODEL_ID = os.environ.get("QWEN_MODEL_ID", "/SSD2/models/Qwen3.5-9B")
BACKEND = os.environ.get("SOCIAL_PROPERTIES_BACKEND", "transformers").lower()
DRY_RUN = False
DRY_RUN = os.environ.get("SOCIAL_PROPERTIES_DRY_RUN", str(DRY_RUN)).lower() in {"1", "true", "yes", "y"}

N_PILOT_EXAMPLES = int(os.environ.get("SOCIAL_PROPERTIES_N", "5"))
RANDOM_SEED = int(os.environ.get("SOCIAL_PROPERTIES_SEED", "17"))
SHOW_RAW_OUTPUTS = os.environ.get("SOCIAL_PROPERTIES_SHOW_RAW", "true").lower() in {"1", "true", "yes", "y"}
RAW_OUTPUT_CHAR_LIMIT = int(os.environ.get("SOCIAL_PROPERTIES_RAW_CHAR_LIMIT", "4000"))
WRITE_FILES = os.environ.get("SOCIAL_PROPERTIES_WRITE_FILES", "false").lower() in {"1", "true", "yes", "y"}
OUTPUT_DIR_NAME = "outputs/social_properties_toy"

random.seed(RANDOM_SEED)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents, Path("/home/jorge/babylm-baby-steps")]
    for path in candidates:
        if (path / "pyproject.toml").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError("Could not find the babylm-baby-steps repo root.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / OUTPUT_DIR_NAME

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"MODEL_ID  = {MODEL_ID}")
print(f"BACKEND   = {BACKEND}")
print(f"DRY_RUN   = {DRY_RUN}")
print(f"CUDA_VISIBLE_DEVICES = {os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>')}")
print(f"N_PILOT_EXAMPLES = {N_PILOT_EXAMPLES}")
print(f"SHOW_RAW_OUTPUTS = {SHOW_RAW_OUTPUTS}")
print(f"WRITE_FILES = {WRITE_FILES}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")


## Backend Abstraction

`generate_text(prompt, max_new_tokens, temperature, top_p)` is the only model-facing function used by the pipeline. The Transformers path is lazy-loaded on first use. The vLLM path is included for easy swapping if `vllm` is available in the kernel.


In [ ]:
DEFAULT_SYSTEM_PROMPT = """
You generate safe, natural English examples about everyday social behavior.
Infer social properties only from behavior in the described scenario.
Never infer a trait from identity, appearance, accent, race, ethnicity, religion, gender, nationality, disability, class markers, or family background.
When asked for JSON, return only valid JSON.
""".strip()

JSON_SYSTEM_PROMPT = """
Return only valid JSON that follows the requested schema.
The first character must be { and the last character must be }.
Use ordinary everyday situations and behavior-based social evidence.
Avoid protected-attribute cues, appearance-based trait inference, stereotypes, and permanent labels from weak evidence.
""".strip()

PROSE_SYSTEM_PROMPT = """
Write natural prose about everyday social behavior.
Use concrete actions and cautious social inferences.
Do not use bullets, labels, benchmark formatting, or moral-quiz wording.
Do not infer social traits from identity or appearance.
""".strip()

EDITOR_SYSTEM_PROMPT = """
Edit or check social-property examples for safety, naturalness, and behavior-grounded inference.
Reject identity-based or appearance-based trait inference.
Soften overconfident permanent labels.
When asked for JSON, return only valid JSON.
""".strip()

_tf_model = None
_tf_tokenizer = None
_tf_processor = None
_vllm_llm = None
_vllm_tokenizer = None
_vllm_sampling_params = None


import atexit
import gc


def release_model_memory(verbose: bool = True) -> None:
    """Drop local model handles and ask PyTorch to release cached CUDA memory."""
    global _tf_model, _tf_tokenizer, _tf_processor
    global _vllm_llm, _vllm_tokenizer, _vllm_sampling_params

    _tf_model = None
    _tf_tokenizer = None
    _tf_processor = None
    _vllm_llm = None
    _vllm_tokenizer = None
    _vllm_sampling_params = None

    gc.collect()
    try:
        if "torch" in sys.modules:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
    except Exception as exc:
        if verbose:
            print(f"Could not fully clear CUDA cache: {exc}")
    if verbose:
        print("Released model references and cleared PyTorch CUDA cache. If nvidia-smi still shows memory, shut down the Jupyter kernel.")


atexit.register(lambda: release_model_memory(verbose=False))


def strip_thinking_text(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S | re.I)
    text = re.sub(r"^\s*Thinking Process:.*?(?=\n\s*[{])", "", text, flags=re.S | re.I)
    return text.strip()


def chat_messages(prompt: str, system_prompt: str | None = None) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt or DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]


def _apply_chat_template(tokenizer_or_processor, messages: list[dict], *, tokenize: bool, return_tensors: str | None = None):
    kwargs = {
        "tokenize": tokenize,
        "add_generation_prompt": True,
    }
    if return_tensors is not None:
        kwargs["return_tensors"] = return_tensors
        kwargs["return_dict"] = True
    try:
        return tokenizer_or_processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer_or_processor.apply_chat_template(messages, **kwargs)


def load_transformers_backend():
    global _tf_model, _tf_tokenizer, _tf_processor
    if _tf_model is not None:
        return _tf_model, _tf_tokenizer, _tf_processor

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_path = Path(MODEL_ID)
    local_files_only = model_path.exists()
    common_kwargs = {
        "trust_remote_code": True,
        "local_files_only": local_files_only,
        "low_cpu_mem_usage": True,
    }
    if torch.cuda.is_available():
        common_kwargs.update(
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
    else:
        common_kwargs.update(torch_dtype=torch.float32)

    try:
        _tf_tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            local_files_only=local_files_only,
        )
        _tf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **common_kwargs)
    except Exception as causal_error:
        print(f"AutoModelForCausalLM load failed, trying AutoModelForMultimodalLM: {causal_error}")
        from transformers import AutoModelForMultimodalLM, AutoProcessor

        _tf_processor = AutoProcessor.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            local_files_only=local_files_only,
        )
        _tf_model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **common_kwargs)

    _tf_model.eval()
    print("Loaded Transformers backend.")
    print(getattr(_tf_model, "hf_device_map", "single device or no explicit device map"))
    return _tf_model, _tf_tokenizer, _tf_processor


def transformers_generate(
    prompt: str,
    *,
    max_new_tokens: int = 800,
    temperature: float = 0.4,
    top_p: float = 0.9,
    system_prompt: str | None = None,
) -> str:
    import torch

    model, tokenizer, processor = load_transformers_backend()
    chat_obj = processor or tokenizer
    messages = chat_messages(prompt, system_prompt)

    inputs = _apply_chat_template(chat_obj, messages, tokenize=True, return_tensors="pt")
    if hasattr(inputs, "to"):
        input_device = next(model.parameters()).device
        inputs = inputs.to(input_device)
    else:
        input_device = next(model.parameters()).device
        inputs = {k: v.to(input_device) if hasattr(v, "to") else v for k, v in inputs.items()}

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
    }
    if temperature > 0:
        generate_kwargs.update(temperature=temperature, top_p=top_p)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)

    input_len = inputs["input_ids"].shape[-1]
    new_tokens = generated_ids[:, input_len:]
    text = chat_obj.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return strip_thinking_text(text)


def load_vllm_backend():
    global _vllm_llm, _vllm_tokenizer, _vllm_sampling_params
    if _vllm_llm is not None:
        return _vllm_llm, _vllm_tokenizer, _vllm_sampling_params

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tensor_parallel_size = int(os.environ.get("VLLM_TENSOR_PARALLEL_SIZE", "1"))
    _vllm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    _vllm_llm = LLM(
        model=MODEL_ID,
        trust_remote_code=True,
        tensor_parallel_size=tensor_parallel_size,
    )
    _vllm_sampling_params = SamplingParams
    print("Loaded vLLM backend.")
    return _vllm_llm, _vllm_tokenizer, _vllm_sampling_params


def vllm_generate(
    prompt: str,
    *,
    max_new_tokens: int = 800,
    temperature: float = 0.4,
    top_p: float = 0.9,
    system_prompt: str | None = None,
) -> str:
    llm, tokenizer, sampling_params_cls = load_vllm_backend()
    messages = chat_messages(prompt, system_prompt)
    chat_prompt = _apply_chat_template(tokenizer, messages, tokenize=False)
    params = sampling_params_cls(
        max_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    outputs = llm.generate([chat_prompt], params)
    return strip_thinking_text(outputs[0].outputs[0].text)


def generate_text(
    prompt: str,
    max_new_tokens: int = 800,
    temperature: float = 0.4,
    top_p: float = 0.9,
    system_prompt: str | None = None,
) -> str:
    if DRY_RUN:
        return mock_generate_text(prompt)
    if BACKEND == "vllm":
        return vllm_generate(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            system_prompt=system_prompt,
        )
    if BACKEND != "transformers":
        raise ValueError(f"Unknown BACKEND={BACKEND!r}; use 'transformers' or 'vllm'.")
    return transformers_generate(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        system_prompt=system_prompt,
    )


def _prompt_value(prompt: str, key: str, default: str = "") -> str:
    match = re.search(rf"^{re.escape(key)}:[ \t]*(.*)$", prompt, flags=re.M)
    return match.group(1).strip() if match else default


def _mock_bool(prompt: str, key: str, default: bool = False) -> bool:
    value = _prompt_value(prompt, key, str(default).lower()).lower()
    return value in {"1", "true", "yes", "y"}


def _mock_behavior_bundle(target: str, evidence_strength: str, agency_subfacet: str = "") -> tuple[str, str, list[str], str]:
    """Return agent, setting, evidence list, and implied prose seed for a dry-run item."""
    repeated = evidence_strength == "repeated_pattern"
    if target == "trustworthy":
        evidence = ["returned borrowed markers on time", "admitted when one cap was missing"]
        if repeated:
            evidence.append("kept doing what they promised during group projects")
        return "Maya", "classroom supply shelf", evidence, "Maya handled the shared supplies carefully and told the truth about the missing cap."
    if target == "untrustworthy":
        evidence = ["promised to return the markers", "kept them and denied borrowing them"]
        if repeated:
            evidence.append("often changed the story when classmates asked about shared supplies")
        return "Maya", "classroom supply shelf", evidence, "Maya kept the markers after promising to return them and changed the story when asked."
    if target == "friendly":
        evidence = ["made room at the table", "asked another student which game they wanted to try"]
        if repeated:
            evidence.append("regularly invited quiet classmates into games")
        return "Maya", "after-school club", evidence, "Maya moved her backpack, made an open seat, and asked which game sounded fun."
    if target == "hostile":
        evidence = ["mocked another player's mistake", "pushed their cards away from the group"]
        if repeated:
            evidence.append("often snapped at people who asked to join")
        return "Maya", "after-school club", evidence, "Maya mocked the mistake and pushed the other player's cards away."
    if target == "warm":
        evidence = ["noticed someone looked upset after losing a game", "sat nearby and listened"]
        if repeated:
            evidence.append("often checked on teammates after hard moments")
        return "Maya", "team practice", evidence, "Maya sat nearby, listened, and helped the teammate rejoin practice."
    if target == "cold":
        evidence = ["ignored a teammate asking for help", "kept sorting supplies without answering"]
        if repeated:
            evidence.append("usually brushed off classmates who needed support")
        return "Maya", "team practice", evidence, "Maya kept sorting supplies and did not answer when the teammate asked for help."
    if target == "dominant":
        if agency_subfacet == "controlling_dominance":
            evidence = ["interrupted before others finished", "assigned roles without asking"]
            return "Maya", "group project", evidence, "Maya interrupted the discussion and assigned roles before others could weigh in."
        evidence = ["suggested a plan when everyone was stuck", "checked whether the group agreed"]
        return "Maya", "group project", evidence, "Maya suggested a plan, checked for agreement, and helped the group choose next steps."
    if target == "submissive":
        evidence = ["waited for someone else to decide", "went along with the choice after mild disagreement"]
        return "Maya", "group project", evidence, "Maya waited for another person to decide and went along after a small disagreement."
    if target == "confident":
        evidence = ["explained their idea clearly", "kept speaking calmly after a small mistake"]
        return "Maya", "group presentation", evidence, "Maya explained the idea clearly and kept speaking calmly after a small mistake."
    if target == "shy":
        evidence = ["spoke very little during the circle", "looked at their notes before answering softly"]
        return "Maya", "club meeting", evidence, "Maya spoke little during the circle and answered softly after checking their notes."
    if target == "introverted":
        evidence = ["chose a quiet table after the game", "read alone during the noisy break"]
        return "Maya", "school club", evidence, "Maya chose a quiet table after the game and read during the noisy break."
    if target == "extroverted":
        evidence = ["started conversations with several classmates", "joined the group game right away"]
        return "Maya", "school club", evidence, "Maya started several conversations and joined the group game right away."
    if target == "boastful":
        evidence = ["talked mostly about their own score", "took credit for the whole project"]
        return "Maya", "science fair table", evidence, "Maya kept talking about their own score and took credit for the whole project."
    if target == "humble":
        evidence = ["shared credit with teammates", "downplayed their own role"]
        return "Maya", "science fair table", evidence, "Maya shared credit with teammates and downplayed their own role."
    if target == "tolerant":
        evidence = ["welcomed someone from a rival game group", "judged their idea fairly"]
        return "Maya", "game club", evidence, "Maya welcomed the rival game-group member and judged the idea fairly."
    if target == "bigoted":
        evidence = ["refused to work with someone from a different game group", "dismissed their idea before hearing it"]
        return "Maya", "game club", evidence, "Maya refused to work with the other game-group member and dismissed the idea before hearing it."
    return "Maya", "club room", ["acted in a clear, observable way"], "Maya acted in a clear, observable way."


def mock_generate_text(prompt: str) -> str:
    """Small deterministic mock so parsing, audit, and export can run without a model."""
    if "SCENARIO_PLAN_REQUEST" in prompt:
        axis = _prompt_value(prompt, "AXIS", "warmth_affiliation")
        target = _prompt_value(prompt, "TARGET_TRAIT", "friendly")
        contrast = _prompt_value(prompt, "CONTRAST_TRAIT", "")
        style_id = _prompt_value(prompt, "STYLE_ID", "everyday_narrative")
        length_bucket = _prompt_value(prompt, "LENGTH_BUCKET", "short_64")
        scenario_id = _prompt_value(prompt, "SCENARIO_ID", f"dry_{random.randint(0, 999999):06d}")
        evidence_strength = _prompt_value(prompt, "EVIDENCE_STRENGTH", "single_moment")
        agency_subfacet = _prompt_value(prompt, "AGENCY_SUBFACET", "")
        label_mode = _prompt_value(prompt, "LABEL_MODE", "implicit")
        trait_word_should_appear = label_mode != "implicit"
        agent, setting, evidence, prose_seed = _mock_behavior_bundle(target, evidence_strength, agency_subfacet)
        wording_options = globals().get("INFERENCE_WORDING", {}).get(evidence_strength, ["came across as {trait}"])
        social_inference = wording_options[0].format(trait=target)
        return json.dumps(
            {
                "scenario_id": scenario_id,
                "axis": axis,
                "target_property": target,
                "contrast_property": contrast,
                "agent": agent,
                "setting": setting,
                "behavior_evidence": evidence,
                "social_inference": social_inference,
                "evidence_strength": evidence_strength,
                "label_mode": label_mode,
                "trait_word_should_appear": trait_word_should_appear,
                "agency_subfacet": agency_subfacet,
                "caveat": "One scene is limited evidence." if evidence_strength == "single_moment" else "Repeated behavior supports a stronger inference.",
                "style_id": style_id,
                "length_bucket": length_bucket,
                "avoid_cues": list(globals().get("AVOID_SOCIAL_CUES", [])),
                "safety_notes": ["Inference is based on observed behavior only."],
                "mock_prose_seed": prose_seed,
            }
        )

    if "RENDER_SOCIAL_PROSE_REQUEST" in prompt:
        scenario_id = _prompt_value(prompt, "SCENARIO_ID", "dry_render")
        axis = _prompt_value(prompt, "AXIS", "warmth_affiliation")
        target = _prompt_value(prompt, "TARGET_TRAIT", "friendly")
        contrast = _prompt_value(prompt, "CONTRAST_TRAIT", "")
        style_id = _prompt_value(prompt, "STYLE_ID", "everyday_narrative")
        length_bucket = _prompt_value(prompt, "LENGTH_BUCKET", "short_64")
        evidence_strength = _prompt_value(prompt, "EVIDENCE_STRENGTH", "single_moment")
        agency_subfacet = _prompt_value(prompt, "AGENCY_SUBFACET", "")
        label_mode = _prompt_value(prompt, "LABEL_MODE", "implicit")
        trait_word_should_appear = label_mode != "implicit"
        agent, setting, evidence, prose_seed = _mock_behavior_bundle(target, evidence_strength, agency_subfacet)
        if label_mode == "evidence_then_label":
            if evidence_strength == "single_moment":
                text = f"During {setting}, {prose_seed} In that moment, {agent} acted in a {target} way."
            else:
                text = f"During {setting}, {prose_seed} Over time, people came to see {agent} as {target}."
        elif label_mode == "label_then_evidence":
            text = f"{agent} seemed {target} during {setting}: {prose_seed} The label was grounded in those actions, not identity."
        else:
            text = f"During {setting}, {prose_seed} The moment gave classmates enough behavior to read the social cue without needing a label."
        return json.dumps(
            {
                "scenario_id": scenario_id,
                "status": "draft",
                "text": text,
                "metadata": {
                    "axis": axis,
                    "target_property": target,
                    "contrast_property": contrast,
                    "style_id": style_id,
                    "length_bucket": length_bucket,
                    "evidence_strength": evidence_strength,
                    "agency_subfacet": agency_subfacet,
                    "label_mode": label_mode,
                    "trait_word_should_appear": trait_word_should_appear,
                    "contains_exact_trait_word": bool(re.search(rf"\\b{re.escape(target)}\\b", text, flags=re.I)),
                    "contains_negative_trait": False,
                },
            }
        )

    if "EDIT_OR_CHECK_SOCIAL_PROSE_REQUEST" in prompt:
        scenario_id = _prompt_value(prompt, "SCENARIO_ID", "dry_edit")
        text = _prompt_value(prompt, "TEXT", "")
        if not text:
            text = "Maya made room at the table and invited another student into the card game."
        return json.dumps(
            {
                "scenario_id": scenario_id,
                "verdict": "keep",
                "text": text,
                "reason": "Behavior-grounded and safe.",
                "metadata": {},
            }
        )

    return '{"ok": true}'


print("generate_text is ready.")


## Optional Smoke Test

Leave this cell skipped if you do not want to load the model yet. In `DRY_RUN` mode it returns a mock JSON object.


In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    smoke = generate_text(
        "Return JSON only: {\"status\":\"ok\", \"message\":\"social pilot ready\"}",
        max_new_tokens=80,
        temperature=0.0,
        top_p=1.0,
        system_prompt=JSON_SYSTEM_PROMPT,
    )
    print(smoke)
else:
    print("Set SMOKE_TEST=True when you want to load/test the backend.")


## 2. Toy Social-Property Ontology

The ontology is deliberately small and behavior-centered. The tolerance axis uses low-stakes fictional or everyday group differences rather than protected-attribute stereotypes.


In [ ]:
SOCIAL_ONTOLOGY = [
    {
        "axis": "warmth_affiliation",
        "anchor_traits": ["friendly", "hostile", "warm", "cold"],
        "evidence_patterns": ["includes others", "comforts", "listens", "ignores", "mocks", "excludes"],
    },
    {
        "axis": "reliability_trust",
        "anchor_traits": ["trustworthy", "untrustworthy"],
        "evidence_patterns": [
            "keeps promises",
            "returns borrowed items",
            "tells the truth",
            "breaks promises",
            "lies about what happened",
        ],
    },
    {
        "axis": "agency_status",
        "anchor_traits": ["dominant", "submissive", "confident"],
        "subfacets": {
            "prosocial_leadership": [
                "organizes the group when everyone is confused",
                "suggests a plan and checks whether others agree",
                "helps assign roles fairly",
                "speaks up so the group can finish the task",
                "takes responsibility when a decision is needed",
            ],
            "controlling_dominance": [
                "orders others around without listening",
                "interrupts to force their own plan",
                "decides roles before anyone else can speak",
                "pushes their choice even after others object",
                "takes control of shared materials without asking",
            ],
            "confident_behavior": [
                "explains their idea clearly",
                "asks a question in front of the group",
                "tries a difficult task without freezing",
                "keeps speaking calmly after a small mistake",
                "volunteers to present the group's work",
            ],
            "yielding_or_deferential_behavior": [
                "lets others choose first",
                "backs down after mild disagreement",
                "waits for someone else to decide",
                "goes along with the group's choice",
                "avoids stating a preference even when asked",
            ],
        },
        "common_confounds": [
            "dominant is not always hostile",
            "confident is not always dominant",
            "submissive or yielding is not always bad",
            "leadership can be warm and cooperative",
            "control can be cold or unfair when it ignores others",
        ],
    },
    {
        "axis": "sociability",
        "anchor_traits": ["shy", "introverted", "extroverted"],
        "evidence_patterns": [
            "starts conversations",
            "seeks groups",
            "avoids attention",
            "prefers quiet",
            "speaks little in a crowd",
        ],
    },
    {
        "axis": "humility_self_presentation",
        "anchor_traits": ["boastful", "humble"],
        "evidence_patterns": ["brags", "takes all credit", "shares credit", "downplays own role", "praises others"],
    },
    {
        "axis": "tolerance_social_attitude",
        "anchor_traits": ["tolerant", "bigoted"],
        "evidence_patterns": [
            "treats someone with different club preferences fairly",
            "treats someone from another game group unfairly",
            "welcomes a rival team fan",
            "refuses to work with someone because of a harmless hobby",
        ],
        "low_stakes_differences": ["clubs", "teams", "hobbies", "schools", "game groups", "favorite snacks"],
    },
]

DISCOURSE_STYLES = [
    {"style_id": "everyday_narrative", "prompt_hint": "Write a compact everyday scene with concrete actions and natural narration."},
    {"style_id": "classroom_observation", "prompt_hint": "Write like a teacher or observer quietly noting what happened in class."},
    {"style_id": "parent_child_conversation", "prompt_hint": "Write a natural parent-child conversation without lesson-like labels."},
    {"style_id": "short_dialogue", "prompt_hint": "Use brief dialogue embedded in prose, without C1/T1 labels."},
    {"style_id": "group_project_note", "prompt_hint": "Write like a concise note about a group project interaction."},
    {"style_id": "neighbor_household_anecdote", "prompt_hint": "Write a small neighbor or household anecdote with ordinary stakes."},
    {"style_id": "before_and_after_description", "prompt_hint": "Describe what the agent did before and after a social moment."},
    {"style_id": "lost_and_found_description", "prompt_hint": "Write about a lost-and-found or borrowed-item situation."},
    {"style_id": "simple_explanation", "prompt_hint": "Give a simple explanation that still sounds like prose, not a worksheet."},
    {"style_id": "school_club_scene", "prompt_hint": "Write a scene from a school club, game group, team, or shared activity."},
]

LENGTH_BUCKETS = {
    "short_64": {"word_range": (35, 60), "description": "about 35-60 words", "max_new_tokens": 180},
    "medium_128": {"word_range": (70, 120), "description": "about 70-120 words", "max_new_tokens": 260},
    "long_256": {"word_range": (140, 220), "description": "about 140-220 words", "max_new_tokens": 420},
}

TRAIT_CONTRASTS = {
    "friendly": "hostile",
    "hostile": "friendly",
    "warm": "cold",
    "cold": "warm",
    "trustworthy": "untrustworthy",
    "untrustworthy": "trustworthy",
    "dominant": "submissive",
    "submissive": "dominant",
    "introverted": "extroverted",
    "extroverted": "introverted",
    "boastful": "humble",
    "humble": "boastful",
    "tolerant": "bigoted",
    "bigoted": "tolerant",
}

TRAIT_NEAR_CONTRASTS = {
    "confident": ["shy", "hesitant", "uncertain"],
    "shy": ["confident", "outgoing"],
    "dominant": ["confident", "assertive"],
    "boastful": ["confident"],
    "introverted": ["shy"],
    "cold": ["introverted"],
}

EVIDENCE_STRENGTHS = ["single_moment", "repeated_pattern"]

INFERENCE_WORDING = {
    "single_moment": [
        "acted in a {trait} way",
        "seemed {trait} in that moment",
        "came across as {trait}",
        "showed a {trait} side",
    ],
    "repeated_pattern": [
        "was often seen as {trait}",
        "people came to see them as {trait}",
        "had a reputation for being {trait}",
        "usually acted {trait}",
    ],
}

LABEL_MODES = ["implicit", "evidence_then_label", "label_then_evidence"]
LABEL_MODE_SEQUENCE = ["implicit", "evidence_then_label", "implicit", "label_then_evidence"]
LABEL_MODE_DESCRIPTIONS = {
    "implicit": "Use behavior only. Do not use the exact target trait word.",
    "evidence_then_label": "Give concrete behavioral evidence first, then use the exact target trait word cautiously.",
    "label_then_evidence": "Use the exact target trait word early, then immediately ground it in concrete behavior.",
}

AVOID_SOCIAL_CUES = [
    "race",
    "ethnicity",
    "religion",
    "gender",
    "nationality",
    "accent",
    "disability",
    "appearance",
    "body size",
    "clothing price",
    "family background",
    "neighborhood",
    "name-based stereotypes",
]

TRAIT_VALENCE = {
    "friendly": "positive",
    "warm": "positive",
    "trustworthy": "positive",
    "confident": "positive",
    "extroverted": "neutral_or_positive",
    "humble": "positive",
    "tolerant": "positive",
    "hostile": "negative",
    "cold": "negative",
    "untrustworthy": "negative",
    "boastful": "negative",
    "bigoted": "negative",
    "dominant": "context_dependent",
    "submissive": "context_dependent",
    "shy": "neutral",
    "introverted": "neutral",
}

AGENCY_SUBFACETS = SOCIAL_ONTOLOGY[2]["subfacets"]
AXIS_BY_ID = {card["axis"]: card for card in SOCIAL_ONTOLOGY}
STYLE_BY_ID = {card["style_id"]: card for card in DISCOURSE_STYLES}
ALL_TRAITS = sorted({trait for card in SOCIAL_ONTOLOGY for trait in card["anchor_traits"]})
STRICT_CONTRAST_TRAITS = set(TRAIT_CONTRASTS)

print(f"Axes: {len(SOCIAL_ONTOLOGY)}")
print(f"Traits: {ALL_TRAITS}")
print(f"Strict contrast traits: {sorted(STRICT_CONTRAST_TRAITS)}")
print(f"Styles: {len(DISCOURSE_STYLES)}")


## 3. Scenario Planning Prompt

`make_social_property_plan(...)` asks the model for one behavior-grounded JSON scenario plan. It intentionally separates behavioral evidence from the inferred social property.


In [ ]:
def choose_agency_subfacet(target_trait: str, index_hint: int = 0) -> str:
    if target_trait == "dominant":
        return ["prosocial_leadership", "controlling_dominance"][index_hint % 2]
    if target_trait == "confident":
        return "confident_behavior"
    if target_trait == "submissive":
        return "yielding_or_deferential_behavior"
    return ""


def make_social_property_plan(
    axis_card: dict,
    target_trait: str,
    contrast_trait: str,
    style_id: str,
    length_bucket: str,
    *,
    evidence_strength: str | None = None,
    label_mode: str | None = None,
    trait_word_should_appear: bool | None = None,
    agency_subfacet: str | None = None,
) -> dict | None:
    evidence_strength = evidence_strength or random.choice(EVIDENCE_STRENGTHS)
    if evidence_strength not in EVIDENCE_STRENGTHS:
        evidence_strength = "single_moment"
    if label_mode is None:
        if trait_word_should_appear is None:
            label_mode = random.choice(LABEL_MODE_SEQUENCE)
        else:
            label_mode = "evidence_then_label" if trait_word_should_appear else "implicit"
    if label_mode not in LABEL_MODES:
        label_mode = "implicit"
    trait_word_should_appear = label_mode != "implicit"
    if axis_card["axis"] != "agency_status":
        agency_subfacet = ""
    else:
        agency_subfacet = agency_subfacet or choose_agency_subfacet(target_trait)
    contrast_trait = contrast_trait or TRAIT_CONTRASTS.get(target_trait, "")
    scenario_id = (
        f"social_{axis_card['axis']}_{target_trait}_{style_id}_{length_bucket}_"
        f"{evidence_strength}_{random.randint(0, 999999):06d}"
    )
    wording_options = INFERENCE_WORDING[evidence_strength]
    prompt = f"""
SCENARIO_PLAN_REQUEST
SCENARIO_ID: {scenario_id}
AXIS: {axis_card['axis']}
TARGET_TRAIT: {target_trait}
CONTRAST_TRAIT: {contrast_trait}
STYLE_ID: {style_id}
LENGTH_BUCKET: {length_bucket}
EVIDENCE_STRENGTH: {evidence_strength}
LABEL_MODE: {label_mode}
TRAIT_WORD_SHOULD_APPEAR: {str(trait_word_should_appear).lower()}
AGENCY_SUBFACET: {agency_subfacet}

Create one EWoK-style social-property scenario plan.

The trait must be inferred from behavior, not identity.
Use ordinary everyday situations: school, family, neighbors, clubs, group projects, sports, chores, lost-and-found, games, shared supplies.
Reject the example if the trait is inferred from identity, appearance, accent, name, nationality, religion, gender, disability, class markers, family background, or neighborhood.
Avoid these cues entirely as trait evidence: {json.dumps(AVOID_SOCIAL_CUES)}
Avoid appearance-based trait inference.
Avoid saying one weak action proves a permanent trait.

Evidence-strength rule:
- If evidence_strength == "single_moment", avoid permanent labels like "is trustworthy" or "is hostile". Prefer wording like: {json.dumps(wording_options)}.
- If evidence_strength == "repeated_pattern", stronger wording is allowed: {json.dumps(wording_options)}.

Trait-label rule:
- label_mode=implicit means the prose should imply the trait through behavior without using the exact target trait label.
- label_mode=evidence_then_label means concrete behavior comes first, followed by a cautious exact trait label.
- label_mode=label_then_evidence means the exact trait label appears early, then concrete behavior immediately grounds it.
- If the exact trait word appears, it must be earned by concrete behavior, not used as a naked label.

Contrast rule:
- Use only contrast_property as a strict contrast.
- TRAIT_NEAR_CONTRASTS are nearby concepts for soft comparisons or audits, not strict opposites.
- Do not treat confident as the strict opposite of shy.
- Do not treat submissive as the strict opposite of confident.

Agency/status rules:
- For dominant, include both prosocial leadership and controlling dominance across examples.
- Do not collapse dominant into bad or bossy.
- Do not collapse confident into dominant.
- Do not collapse submissive into shy.

Axis card:
{json.dumps(axis_card, indent=2)}

Use exactly this schema:
{{
  "scenario_id": "{scenario_id}",
  "axis": "{axis_card['axis']}",
  "target_property": "{target_trait}",
  "contrast_property": "{contrast_trait}",
  "agent": "...",
  "setting": "...",
  "behavior_evidence": [],
  "social_inference": "...",
  "evidence_strength": "{evidence_strength}",
  "label_mode": "{label_mode}",
  "trait_word_should_appear": {str(trait_word_should_appear).lower()},
  "agency_subfacet": "{agency_subfacet}",
  "caveat": "",
  "style_id": "{style_id}",
  "length_bucket": "{length_bucket}",
  "avoid_cues": [],
  "safety_notes": []
}}
""".strip()

    obj = json_call(
        prompt,
        label=f"plan:{scenario_id}",
        system_prompt=JSON_SYSTEM_PROMPT,
        max_new_tokens=1100,
        temperature=0.35,
        top_p=0.9,
    )
    if obj is None:
        return None

    obj.setdefault("scenario_id", scenario_id)
    obj["axis"] = axis_card["axis"]
    obj["target_property"] = target_trait
    obj["contrast_property"] = contrast_trait
    obj["style_id"] = style_id
    obj["length_bucket"] = length_bucket
    obj["evidence_strength"] = evidence_strength
    obj["label_mode"] = label_mode
    obj["trait_word_should_appear"] = trait_word_should_appear
    obj["agency_subfacet"] = agency_subfacet or ""
    obj.setdefault("behavior_evidence", [])
    obj.setdefault("avoid_cues", list(AVOID_SOCIAL_CUES))
    obj.setdefault("safety_notes", [])
    return obj


## 4. Robust JSON Parsing

These helpers handle fenced JSON, extra preambles, trailing commas, and one retry with a stricter prompt. Raw failures are kept in `RAW_OUTPUT_LOG` for inspection.


In [ ]:
RAW_OUTPUT_LOG = []
MODEL_CALL_LOG = []


def extract_json_object(text: str) -> str | None:
    """Extract the first balanced JSON object string from a model response."""
    if text is None:
        return None
    text = str(text).strip()
    fenced = [
        match.group(1).strip()
        for match in re.finditer(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    ]
    candidates = fenced + [text]

    for candidate in candidates:
        start = candidate.find("{")
        if start < 0:
            continue
        depth = 0
        in_string = False
        escape = False
        for index in range(start, len(candidate)):
            char = candidate[index]
            if in_string:
                if escape:
                    escape = False
                elif char == "\\":
                    escape = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    return candidate[start : index + 1]
    return None


def repair_json_once(text: str) -> str | None:
    """Apply conservative string repairs and return one JSON-object candidate."""
    candidate = extract_json_object(text)
    if candidate is None:
        return None
    repaired = candidate.strip()
    repaired = repaired.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'")
    repaired = re.sub(r",\s*([}\]])", r"\1", repaired)
    repaired = re.sub(r"\bTrue\b", "true", repaired)
    repaired = re.sub(r"\bFalse\b", "false", repaired)
    repaired = re.sub(r"\bNone\b", "null", repaired)
    return repaired


def safe_json_loads(text: str) -> dict | None:
    candidates = []
    if text is not None:
        candidates.append(str(text).strip())
    extracted = extract_json_object(text)
    if extracted is not None:
        candidates.append(extracted)
    repaired = repair_json_once(text)
    if repaired is not None:
        candidates.append(repaired)

    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


def json_call(
    prompt: str,
    *,
    label: str,
    system_prompt: str,
    max_new_tokens: int = 900,
    temperature: float = 0.35,
    top_p: float = 0.9,
    retries: int = 1,
) -> dict | None:
    current_prompt = prompt
    for attempt in range(retries + 1):
        raw = generate_text(
            current_prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature if attempt == 0 else 0.1,
            top_p=top_p,
            system_prompt=system_prompt,
        )
        obj = safe_json_loads(raw)
        call_record = {
            "label": label,
            "attempt": attempt,
            "parse_ok": obj is not None,
            "prompt": current_prompt,
            "raw": raw,
        }
        if obj is not None:
            call_record["parsed"] = obj
        MODEL_CALL_LOG.append(call_record)
        if obj is not None:
            return obj
        RAW_OUTPUT_LOG.append(call_record)
        current_prompt = (
            prompt
            + "\n\nYour previous output was not valid JSON. Return the same requested object again. "
            + "Return only one JSON object, starting with { and ending with }."
        )
    print(f"Skipping {label}: could not parse JSON after {retries + 1} attempt(s).")
    return None

def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
        f.write("\n")


def write_jsonl(path: Path, records) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


print("JSON helpers are ready.")


## 5. Prose Rendering Prompt

`render_social_prose(...)` turns one scenario plan into natural prose plus metadata. The text field should be plain prose only.


In [ ]:
def contains_trait_word(text: str, trait: str) -> bool:
    if not trait:
        return False
    return bool(re.search(rf"\b{re.escape(trait)}\b", text or "", flags=re.I))


def render_social_prose(scenario_plan: dict, style_card: dict) -> dict | None:
    scenario_id = scenario_plan.get("scenario_id", f"missing_id_{random.randint(0, 999999):06d}")
    target_trait = scenario_plan.get("target_property", "")
    contrast_trait = scenario_plan.get("contrast_property", "")
    length_bucket = scenario_plan.get("length_bucket", "short_64")
    length_info = LENGTH_BUCKETS.get(length_bucket, LENGTH_BUCKETS["short_64"])
    evidence_strength = scenario_plan.get("evidence_strength", "single_moment")
    agency_subfacet = scenario_plan.get("agency_subfacet", "")
    label_mode = scenario_plan.get("label_mode") or ("evidence_then_label" if scenario_plan.get("trait_word_should_appear") else "implicit")
    if label_mode not in LABEL_MODES:
        label_mode = "implicit"
    trait_word_should_appear = label_mode != "implicit"
    wording_options = INFERENCE_WORDING.get(evidence_strength, INFERENCE_WORDING["single_moment"])

    prompt = f"""
RENDER_SOCIAL_PROSE_REQUEST
SCENARIO_ID: {scenario_id}
AXIS: {scenario_plan.get('axis', '')}
TARGET_TRAIT: {target_trait}
CONTRAST_TRAIT: {contrast_trait}
STYLE_ID: {style_card.get('style_id', '')}
LENGTH_BUCKET: {length_bucket}
EVIDENCE_STRENGTH: {evidence_strength}
LABEL_MODE: {label_mode}
TRAIT_WORD_SHOULD_APPEAR: {str(trait_word_should_appear).lower()}
AGENCY_SUBFACET: {agency_subfacet}

Render the scenario plan as natural prose.

Requirements:
- Return only valid JSON. First character must be {{, last character must be }}.
- Write natural prose only inside the text field.
- No bullets, labels, tables, benchmark format, or C1/T1 formatting.
- Do not sound like a moral quiz or worksheet.
- Do not over-explain every inference.
- Do not add identity-based explanations.
- The trait must follow from behavior in the scene, not identity, appearance, accent, name, nationality, religion, gender, disability, class markers, family background, or neighborhood.
- Avoid these cues as evidence: {json.dumps(AVOID_SOCIAL_CUES)}
- Follow label_mode exactly.
- label_mode=implicit: imply the target property without using the exact target trait word.
- label_mode=evidence_then_label: describe concrete behavior first, then use the exact target trait word cautiously.
- label_mode=label_then_evidence: use the exact target trait word early, then immediately explain it with concrete behavior.
- Exact trait labels are allowed in the two explicit modes, but never use the label without behavior that earns it.
- If evidence_strength is single_moment, avoid permanent labels like "is {target_trait}". Prefer: {json.dumps(wording_options)}.
- If evidence_strength is repeated_pattern, stronger wording is allowed but should still be behavior-grounded.
- Do not collapse quiet or introverted behavior into coldness.
- Do not turn confidence into boastfulness unless the person is actually bragging or taking undue credit.
- Do not turn dominance into hostility unless the behavior is controlling or cold.
- Vary openings.
- Avoid overusing "because", "therefore", "this shows", "this proves", and "that means".
- Target length: {length_info['description']}.

Style card:
{json.dumps(style_card, indent=2)}

Scenario plan:
{json.dumps(scenario_plan, indent=2)}

Use exactly this schema:
{{
  "scenario_id": "{scenario_id}",
  "status": "draft",
  "text": "...",
  "metadata": {{
    "axis": "{scenario_plan.get('axis', '')}",
    "target_property": "{target_trait}",
    "contrast_property": "{contrast_trait}",
    "style_id": "{style_card.get('style_id', '')}",
    "length_bucket": "{length_bucket}",
    "evidence_strength": "{evidence_strength}",
    "agency_subfacet": "{agency_subfacet}",
    "label_mode": "{label_mode}",
    "trait_word_should_appear": {str(trait_word_should_appear).lower()},
    "contains_exact_trait_word": false,
    "contains_negative_trait": false
  }}
}}
""".strip()

    obj = json_call(
        prompt,
        label=f"render:{scenario_id}",
        system_prompt=PROSE_SYSTEM_PROMPT,
        max_new_tokens=length_info["max_new_tokens"] + 220,
        temperature=0.65,
        top_p=0.92,
    )
    if obj is None:
        return None

    text = str(obj.get("text", "")).strip()
    if not text:
        RAW_OUTPUT_LOG.append({"label": f"render_empty:{scenario_id}", "attempt": 0, "raw": obj})
        return None

    metadata = dict(obj.get("metadata", {}))
    metadata.update(
        {
            "axis": scenario_plan.get("axis"),
            "target_property": target_trait,
            "contrast_property": contrast_trait,
            "style_id": style_card.get("style_id"),
            "length_bucket": length_bucket,
            "evidence_strength": evidence_strength,
            "agency_subfacet": agency_subfacet,
            "label_mode": label_mode,
            "trait_word_should_appear": trait_word_should_appear,
            "contains_exact_trait_word": contains_trait_word(text, target_trait),
            "contains_negative_trait": any(
                contains_trait_word(text, trait)
                for trait, valence in TRAIT_VALENCE.items()
                if valence == "negative"
            ),
        }
    )
    return {
        "scenario_id": scenario_id,
        "status": "draft",
        "text": text,
        "metadata": metadata,
        "scenario_plan": scenario_plan,
    }


## 6. Edit / Check Prompt

The editor preserves behavior-grounded inferences, softens overconfident language, and rejects unsafe identity- or appearance-based examples.


In [ ]:
def edit_or_check_social_prose(example: dict) -> dict:
    scenario_id = example.get("scenario_id", f"missing_id_{random.randint(0, 999999):06d}")
    prompt = f"""
EDIT_OR_CHECK_SOCIAL_PROSE_REQUEST
SCENARIO_ID: {scenario_id}
TEXT: {example.get('text', '')}

Review this EWoK-style social-property example.

Editor requirements:
- Preserve the intended social-property inference.
- Reject protected-attribute or identity-based trait inference.
- Reject the example if the trait is inferred from identity, appearance, accent, name, nationality, religion, gender, disability, class markers, family background, or neighborhood.
- The trait must be inferred from behavior in the scene.
- Respect metadata.label_mode.
- If label_mode is implicit, remove the exact target trait word and let behavior carry the inference.
- If label_mode is evidence_then_label, exact trait wording is allowed after concrete behavioral evidence.
- If label_mode is label_then_evidence, exact trait wording is allowed early, but concrete behavior must immediately ground it.
- Do not treat exact trait labels as errors when label_mode requests them; only revise naked or overconfident labels.
- Soften overconfident permanent trait labels, especially when evidence_strength is single_moment.
- Prefer "acted in a trustworthy way" or "seemed cold in that moment" when evidence is limited.
- Avoid moralizing phrasing like "good person" or "bad person".
- Remove "this proves", "that means", and "therefore" when the tone becomes worksheet-like.
- Avoid turning every story into a lesson.
- Avoid turning quiet or introverted behavior into coldness.
- Avoid turning confidence into boastfulness unless the person is actually bragging or taking undue credit.
- Avoid turning dominance into hostility unless the behavior is controlling or cold.
- Avoid adding new facts.

Return only valid JSON with this schema:
{{
  "scenario_id": "{scenario_id}",
  "verdict": "keep",
  "text": "...",
  "reason": "...",
  "metadata": {{}}
}}

Use verdict "keep", "revise", or "reject".

Example:
{json.dumps(example, indent=2)}
""".strip()

    obj = json_call(
        prompt,
        label=f"edit:{scenario_id}",
        system_prompt=EDITOR_SYSTEM_PROMPT,
        max_new_tokens=950,
        temperature=0.2,
        top_p=0.9,
    )
    if obj is None:
        return {
            "scenario_id": scenario_id,
            "verdict": "reject",
            "text": example.get("text", ""),
            "reason": "Unparseable editor output.",
            "metadata": dict(example.get("metadata", {})),
            "scenario_plan": example.get("scenario_plan", {}),
        }

    verdict = str(obj.get("verdict", "keep")).strip().lower()
    if verdict not in {"keep", "revise", "reject"}:
        verdict = "revise"

    text = str(obj.get("text") or example.get("text", "")).strip()
    metadata = dict(example.get("metadata", {}))
    metadata.update(obj.get("metadata", {}) if isinstance(obj.get("metadata"), dict) else {})
    plan = example.get("scenario_plan", {}) if isinstance(example.get("scenario_plan"), dict) else {}
    metadata.setdefault("label_mode", plan.get("label_mode", "implicit"))
    metadata.setdefault("trait_word_should_appear", metadata.get("label_mode") != "implicit")
    metadata["editor_verdict"] = verdict
    metadata["editor_reason"] = str(obj.get("reason", "")).strip()
    metadata["contains_exact_trait_word"] = contains_trait_word(text, metadata.get("target_property", ""))
    metadata["contains_negative_trait"] = any(
        contains_trait_word(text, trait)
        for trait, valence in TRAIT_VALENCE.items()
        if valence == "negative"
    )

    return {
        "scenario_id": scenario_id,
        "verdict": verdict,
        "text": text,
        "reason": str(obj.get("reason", "")).strip(),
        "metadata": metadata,
        "scenario_plan": example.get("scenario_plan", {}),
    }


## 7. Programmatic Audit

Most audit issues are warnings. Hard drops are reserved for exact duplicates, unparseable rows, protected-attribute leakage/stereotyping, and identity- or appearance-based trait inference.


In [ ]:
CONNECTIVE_PATTERNS = [
    "because",
    "so",
    "therefore",
    "unlike",
    "instead of",
    "rather than",
    "this shows",
    "this proves",
    "that means",
]

OVERCONFIDENT_PATTERNS = [
    "always",
    "never",
    "bad person",
    "good person",
    "proves",
    "everyone knew",
]

SAFETY_CUE_RE = re.compile(
    r"\b("
    r"race|racial|ethnic|ethnicity|religion|religious|muslim|christian|jewish|hindu|buddhist|"
    r"gender|male|female|boy|girl|man|woman|trans|nonbinary|gay|lesbian|nationality|immigrant|"
    r"accent|disabled|disability|wheelchair|blind|deaf|autistic|appearance|body size|clothing price|"
    r"poor|rich|social class|working class|middle class|upper class|lower class|class marker|"
    r"skin|family background|neighborhood|name-based stereotype|name based stereotype"
    r")\b",
    flags=re.I,
)

PROTECTED_ATTRIBUTE_RE = SAFETY_CUE_RE

APPEARANCE_RE = re.compile(
    r"\b(appearance|face|clothes|clothing|hair|skin|tall|short|pretty|handsome|ugly|body size|accent)\b",
    flags=re.I,
)

IDENTITY_INFERENCE_RE = re.compile(
    r"\b(because|since|as|due to)\b.{0,100}\b("
    r"race|ethnicity|religion|gender|nationality|accent|appearance|looked|looks|disabled|disability|"
    r"family|background|neighborhood|poor|rich|skin|name"
    r")\b",
    flags=re.I | re.S,
)

PRONOUN_RE = re.compile(r"\b(he|him|his|she|her|hers|they|them|their|theirs)\b", flags=re.I)
NAME_RE = re.compile(r"\b[A-Z][a-z]{2,}\b")

STOP_CAPITALIZED = {
    "At", "After", "Before", "During", "In", "On", "When", "While", "The", "A", "An", "This", "That",
    "Later", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
}


def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def words(text: str) -> list[str]:
    return re.findall(r"[a-z0-9']+", text.lower())


def first_n_words(text: str, n: int) -> str:
    return " ".join(words(text)[:n])


def jaccard_similarity(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def add_flag(row: dict, code: str, severity: str, message: str) -> None:
    row.setdefault("audit_flags", []).append({"code": code, "severity": severity, "message": message})


def connective_counts(text: str) -> dict[str, int]:
    lowered = text.lower()
    return {pattern: len(re.findall(rf"\b{re.escape(pattern)}\b", lowered)) for pattern in CONNECTIVE_PATTERNS}


def possible_names(text: str) -> list[str]:
    return [name for name in NAME_RE.findall(text) if name not in STOP_CAPITALIZED]


def length_bucket_for_word_count(count: int) -> str:
    if count <= 65:
        return "short_64"
    if count <= 130:
        return "medium_128"
    return "long_256"


def scenario_safety_text(row: dict) -> str:
    plan = row.get("scenario_plan", {}) if isinstance(row.get("scenario_plan"), dict) else {}
    content = {
        "text": row.get("text", ""),
        "agent": plan.get("agent", ""),
        "setting": plan.get("setting", ""),
        "behavior_evidence": plan.get("behavior_evidence", []),
        "social_inference": plan.get("social_inference", ""),
    }
    return json.dumps(content, ensure_ascii=False)


def template_match_code(text: str, target_trait: str) -> str | None:
    lowered = text.lower()
    if target_trait == "trustworthy" and ("returned the lost wallet" in lowered or "lost wallet" in lowered):
        return "trustworthy_returned_wallet"
    if target_trait == "boastful" and ("bragged about" in lowered and "trophy" in lowered):
        return "boastful_trophy_brag"
    if target_trait == "shy" and ("hid in the corner" in lowered or "hiding in the corner" in lowered):
        return "shy_hid_in_corner"
    if target_trait == "dominant" and ("gave orders" in lowered or "ordered everyone" in lowered):
        return "dominant_gave_orders"
    return None


def uses_is_trait(text: str, trait: str) -> bool:
    if not trait:
        return False
    return bool(re.search(rf"\b(is|was|were)\s+(?:very\s+|really\s+|quite\s+)?{re.escape(trait)}\b", text, flags=re.I))


def trait_word_index(text: str, trait: str) -> int | None:
    if not trait:
        return None
    text_words = words(text)
    trait_words = words(trait)
    if not trait_words:
        return None
    width = len(trait_words)
    for idx in range(0, len(text_words) - width + 1):
        if text_words[idx : idx + width] == trait_words:
            return idx
    return None


def audit_social_examples(examples: list[dict]) -> tuple[list[dict], dict]:
    audited = []
    normalized_seen = {}
    opening5 = Counter()
    opening8 = Counter()
    opening12 = Counter()
    text_sets = []
    template_matches = defaultdict(list)

    for index, original in enumerate(examples):
        row = copy.deepcopy(original)
        row.setdefault("scenario_id", row.get("id", f"row_{index:04d}"))
        row["audit_flags"] = []
        row["audit_status"] = "keep"
        text = row.get("text")

        if not isinstance(text, str) or not text.strip():
            add_flag(row, "unparseable_or_empty_text", "hard_drop", "Missing or non-string text.")
            row["audit_status"] = "drop"
            audited.append(row)
            text_sets.append(set())
            continue

        metadata = row.setdefault("metadata", {})
        plan = row.get("scenario_plan", {}) if isinstance(row.get("scenario_plan"), dict) else {}
        target_trait = str(metadata.get("target_property") or plan.get("target_property") or "")
        contrast_trait = str(metadata.get("contrast_property") or plan.get("contrast_property") or "")
        evidence_strength = str(metadata.get("evidence_strength") or plan.get("evidence_strength") or "single_moment")
        agency_subfacet = str(metadata.get("agency_subfacet") or plan.get("agency_subfacet") or "")
        label_mode = str(metadata.get("label_mode") or plan.get("label_mode") or "")
        if label_mode not in LABEL_MODES:
            legacy_trait_word = bool(metadata.get("trait_word_should_appear") or plan.get("trait_word_should_appear"))
            label_mode = "evidence_then_label" if legacy_trait_word else "implicit"
        trait_word_should_appear = label_mode != "implicit"
        metadata["target_property"] = target_trait
        metadata["contrast_property"] = contrast_trait
        metadata["evidence_strength"] = evidence_strength
        metadata["agency_subfacet"] = agency_subfacet
        metadata["label_mode"] = label_mode
        metadata["trait_word_should_appear"] = trait_word_should_appear

        clean = normalize_text(text)
        if clean in normalized_seen:
            add_flag(row, "exact_normalized_duplicate", "hard_drop", f"Exact normalized duplicate of {normalized_seen[clean]}.")
        else:
            normalized_seen[clean] = row["scenario_id"]

        safety_text = scenario_safety_text(row)
        if IDENTITY_INFERENCE_RE.search(safety_text):
            add_flag(
                row,
                "protected_attribute_or_identity_evidence",
                "hard_drop",
                "Trait appears tied to protected, identity, appearance, class, family, neighborhood, or name cues.",
            )
        elif SAFETY_CUE_RE.search(safety_text):
            add_flag(
                row,
                "safety_cue_present",
                "warn",
                "Protected, identity, appearance, class, family, or neighborhood cue appears; inspect whether it is evidence or only setting text.",
            )
        if APPEARANCE_RE.search(safety_text) and any(contains_trait_word(text, t) for t in ALL_TRAITS):
            add_flag(row, "appearance_based_trait_inference", "hard_drop", "Trait may be tied to appearance rather than behavior.")

        if target_trait == "shy" and contrast_trait == "confident":
            add_flag(row, "trait_contrast_sanity", "warn", "confident is a near contrast for shy, not a strict contrast.")
        if target_trait == "confident" and contrast_trait in {"shy", "submissive"}:
            add_flag(row, "trait_contrast_sanity", "warn", f"{contrast_trait} should not be a strict contrast for confident.")
        if target_trait == "submissive" and contrast_trait == "confident":
            add_flag(row, "trait_contrast_sanity", "warn", "confident should not be the strict contrast for submissive.")

        lowered = text.lower()
        for pattern in OVERCONFIDENT_PATTERNS:
            if re.search(rf"\b{re.escape(pattern)}\b", lowered):
                add_flag(row, "overconfident_language", "warn", f"Contains overconfident phrase: {pattern}.")

        if evidence_strength == "single_moment":
            if re.search(r"\b(is always|was always|were always|everyone knew they were|everyone knew .* was|bad person|good person)\b", lowered):
                add_flag(row, "single_moment_permanent_label", "warn", "Single-moment evidence uses permanent or global wording.")
            if uses_is_trait(text, target_trait):
                add_flag(row, "single_moment_is_trait", "warn", "Single-moment evidence uses 'is/was [trait]' wording.")

        conn = connective_counts(text)
        if conn.get("therefore", 0) or conn.get("this proves", 0) or conn.get("that means", 0) or conn.get("this shows", 0):
            add_flag(row, "worksheet_connective", "warn", "Contains therefore/this shows/this proves/that means phrasing.")
        if sum(conn.values()) >= 4:
            add_flag(row, "connective_heavy", "warn", "Uses many explicit connective phrases.")

        exact_trait = contains_trait_word(text, target_trait) if target_trait else False
        metadata["contains_exact_trait_word"] = exact_trait
        label_idx = trait_word_index(text, target_trait) if exact_trait else None
        if label_mode == "implicit" and exact_trait:
            add_flag(row, "unexpected_exact_trait_label", "warn", "Plan requested implication, but text contains the exact target trait.")
        if label_mode != "implicit" and not exact_trait:
            add_flag(row, "missing_requested_trait_label", "warn", "Explicit label mode requested the exact trait word, but text does not contain it.")
        if label_mode == "evidence_then_label" and label_idx is not None and label_idx < 12:
            add_flag(row, "label_before_evidence", "warn", "evidence_then_label used the exact trait word before enough behavioral evidence.")
        if label_mode == "label_then_evidence" and label_idx is not None:
            text_word_count = len(words(text))
            if label_idx > max(12, text_word_count // 4):
                add_flag(row, "label_too_late", "warn", "label_then_evidence should place the exact trait word early, then ground it in behavior.")

        word_count = len(words(text))
        metadata["word_count"] = word_count
        expected_bucket = str(metadata.get("length_bucket", ""))
        if expected_bucket in LENGTH_BUCKETS:
            low, high = LENGTH_BUCKETS[expected_bucket]["word_range"]
            if word_count < low or word_count > high:
                add_flag(row, "length_bucket_mismatch", "warn", f"{word_count} words outside {expected_bucket} range {low}-{high}.")
        metadata["observed_length_bucket"] = length_bucket_for_word_count(word_count)

        tcode = template_match_code(text, target_trait)
        if tcode is not None:
            template_matches[tcode].append(row["scenario_id"])

        if any(flag["severity"] == "hard_drop" for flag in row["audit_flags"]):
            row["audit_status"] = "drop"

        audited.append(row)
        opening5[first_n_words(text, 5)] += 1
        opening8[first_n_words(text, 8)] += 1
        opening12[first_n_words(text, 12)] += 1
        text_sets.append(set(words(text)))

    for row in audited:
        if row.get("audit_status") == "drop" or not row.get("text"):
            continue
        op5 = first_n_words(row["text"], 5)
        op8 = first_n_words(row["text"], 8)
        op12 = first_n_words(row["text"], 12)
        if op5 and opening5[op5] > 1:
            add_flag(row, "repeated_first_5_words", "warn", f"Opening 5 words repeated {opening5[op5]} times.")
        if op8 and opening8[op8] > 1:
            add_flag(row, "repeated_first_8_words", "warn", f"Opening 8 words repeated {opening8[op8]} times.")
        if op12 and opening12[op12] > 1:
            add_flag(row, "repeated_first_12_words", "warn", f"Opening 12 words repeated {opening12[op12]} times.")

    for i in range(len(audited)):
        for j in range(i + 1, len(audited)):
            if audited[i].get("audit_status") == "drop" or audited[j].get("audit_status") == "drop":
                continue
            sim = jaccard_similarity(text_sets[i], text_sets[j])
            if sim > 0.80:
                add_flag(audited[i], "near_duplicate_jaccard", "warn", f"Near duplicate of {audited[j]['scenario_id']} (Jaccard {sim:.2f}).")
                add_flag(audited[j], "near_duplicate_jaccard", "warn", f"Near duplicate of {audited[i]['scenario_id']} (Jaccard {sim:.2f}).")

    trait_counts = Counter(row.get("metadata", {}).get("target_property") for row in audited if row.get("metadata"))
    for code, ids in template_matches.items():
        trait = code.split("_")[0]
        if len(ids) >= 2 and len(ids) / max(1, trait_counts.get(trait, 0)) > 0.50:
            for row in audited:
                if row["scenario_id"] in ids:
                    add_flag(row, "behavior_template_collapse", "warn", f"Template pattern overused: {code}.")

    for row in audited:
        if row.get("audit_status") != "drop" and row.get("audit_flags"):
            row["audit_status"] = "warn"

    kept_rows = [row for row in audited if row.get("audit_status") != "drop"]
    all_text = "\n".join(row.get("text", "") for row in kept_rows)
    metadata_rows = [row.get("metadata", {}) for row in audited]
    kept_metadata = [row.get("metadata", {}) for row in kept_rows]
    exact_label_rows = [m for m in kept_metadata if m.get("contains_exact_trait_word")]
    flag_counts = Counter(flag["code"] for row in audited for flag in row.get("audit_flags", []))
    hard_drop_counts = Counter(flag["code"] for row in audited for flag in row.get("audit_flags", []) if flag["severity"] == "hard_drop")
    connective_total = Counter()
    for row in kept_rows:
        connective_total.update(connective_counts(row.get("text", "")))

    pronouns = Counter(match.group(1).lower() for match in PRONOUN_RE.finditer(all_text))
    names = Counter(name for row in kept_rows for name in possible_names(row.get("text", "")))

    def density_by(key: str) -> dict[str, float]:
        groups = defaultdict(list)
        for meta in kept_metadata:
            groups[str(meta.get(key, "<missing>"))].append(bool(meta.get("contains_exact_trait_word")))
        return {group: sum(vals) / max(1, len(vals)) for group, vals in groups.items()}

    single_moment_rows = [row for row in kept_rows if row.get("metadata", {}).get("evidence_strength") == "single_moment"]
    single_moment_is_trait_rate = sum(
        uses_is_trait(row.get("text", ""), row.get("metadata", {}).get("target_property", "")) for row in single_moment_rows
    ) / max(1, len(single_moment_rows))

    dominant_rows = [row for row in kept_rows if row.get("metadata", {}).get("target_property") == "dominant"]
    dominant_subfacets = Counter(row.get("metadata", {}).get("agency_subfacet", "<missing>") for row in dominant_rows)
    controlling_dominant_rate = dominant_subfacets.get("controlling_dominance", 0) / max(1, len(dominant_rows))

    report = {
        "total_examples": len(audited),
        "kept_examples": len([row for row in audited if row.get("audit_status") == "keep"]),
        "warned_examples": len([row for row in audited if row.get("audit_status") == "warn"]),
        "dropped_examples": len([row for row in audited if row.get("audit_status") == "drop"]),
        "style_distribution": dict(Counter(m.get("style_id", "<missing>") for m in metadata_rows)),
        "length_distribution": dict(Counter(m.get("observed_length_bucket", "<missing>") for m in metadata_rows)),
        "planned_length_distribution": dict(Counter(m.get("length_bucket", "<missing>") for m in metadata_rows)),
        "axis_distribution": dict(Counter(m.get("axis", "<missing>") for m in metadata_rows)),
        "target_trait_distribution": dict(Counter(m.get("target_property", "<missing>") for m in metadata_rows)),
        "trait_valence_distribution": dict(Counter(TRAIT_VALENCE.get(m.get("target_property"), "<missing>") for m in metadata_rows)),
        "evidence_strength_distribution": dict(Counter(m.get("evidence_strength", "<missing>") for m in metadata_rows)),
        "label_mode_distribution": dict(Counter(m.get("label_mode", "<missing>") for m in metadata_rows)),
        "agency_subfacet_distribution": dict(Counter(m.get("agency_subfacet", "<none>") or "<none>" for m in metadata_rows)),
        "agency_subfacet_balance": dict(Counter(m.get("agency_subfacet", "<none>") for m in metadata_rows if m.get("axis") == "agency_status")),
        "dominant_subfacet_distribution": dict(dominant_subfacets),
        "controlling_dominant_rate": controlling_dominant_rate,
        "exact_trait_label_density": len(exact_label_rows) / max(1, len(kept_metadata)),
        "exact_trait_label_target": {"target": 0.50, "warn_below": 0.35, "warn_above": 0.65},
        "exact_trait_label_density_by_axis": density_by("axis"),
        "exact_trait_label_density_by_target_trait": density_by("target_property"),
        "single_moment_is_trait_rate": single_moment_is_trait_rate,
        "connective_counts": dict(connective_total),
        "connective_rate_per_example": {key: value / max(1, len(kept_rows)) for key, value in connective_total.items()},
        "pronoun_counts": dict(pronouns),
        "possible_name_counts": dict(names.most_common(30)),
        "top_repeated_opening_5": opening5.most_common(15),
        "top_repeated_opening_8": opening8.most_common(15),
        "audit_flag_counts": dict(flag_counts),
        "hard_drop_counts": dict(hard_drop_counts),
        "safety_flag_counts": {key: value for key, value in flag_counts.items() if "identity" in key or "protected" in key or "appearance" in key},
        "raw_output_failures": len(RAW_OUTPUT_LOG),
    }

    recommendations = []
    if report["dropped_examples"]:
        recommendations.append("Inspect hard drops before scaling; identity/protected-attribute drops should be treated as prompt failures.")
    if report["exact_trait_label_density"] < 0.35:
        recommendations.append("Exact trait labels are sparse; target about 50% explicit labels with 50% implicit examples.")
    if report["exact_trait_label_density"] > 0.65:
        recommendations.append("Exact trait labels are dense; target about 50% explicit labels and increase implicit behavior-only examples.")
    if report["single_moment_is_trait_rate"] > 0.25:
        recommendations.append("Single-moment examples often use 'is [trait]' wording; soften to acted/seemed/came across.")
    if controlling_dominant_rate > 0.75 and len(dominant_rows) >= 3:
        recommendations.append("Dominant examples are overwhelmingly controlling/negative; add prosocial leadership examples.")
    if any(value > 0.40 for value in report["connective_rate_per_example"].values()):
        recommendations.append("Some connective phrases are frequent; ask for more varied causal wording.")
    if flag_counts.get("behavior_template_collapse"):
        recommendations.append("Behavior-template collapse detected; diversify evidence patterns for repeated traits.")
    report["recommendations"] = recommendations
    return audited, report


print("Audit helpers are ready.")


## 8. Inspectable Tiny Pilot

The pilot is split into small notebook-native stages. Run one cell, inspect what the model produced, then continue. Nothing writes corpus artifacts until the final export cell.


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None


def show(obj):
    if "display" in globals():
        display(obj)
    else:
        print(obj)


def as_table(counter_or_dict, key_name: str = "key", value_name: str = "count"):
    items = counter_or_dict.items() if isinstance(counter_or_dict, dict) else counter_or_dict
    rows = [{key_name: key, value_name: value} for key, value in items]
    if pd is None:
        return rows
    return pd.DataFrame(rows)


def shorten_text(text: str, limit: int = RAW_OUTPUT_CHAR_LIMIT) -> str:
    text = str(text)
    if limit <= 0 or len(text) <= limit:
        return text
    return text[:limit].rstrip() + f"\n... <truncated {len(text) - limit} chars>"


def model_calls(prefix: str | None = None) -> list[dict]:
    calls = globals().get("MODEL_CALL_LOG", [])
    if prefix is None:
        return list(calls)
    return [record for record in calls if str(record.get("label", "")).startswith(prefix)]


def print_model_outputs(prefix: str | None = None, limit: int | None = None, char_limit: int = RAW_OUTPUT_CHAR_LIMIT) -> None:
    records = model_calls(prefix)
    if limit is not None:
        records = records[-limit:]
    if not records:
        print(f"No model outputs for prefix={prefix!r}.")
        return
    for i, record in enumerate(records, 1):
        print("=" * 88)
        print(f"{i}. {record.get('label')} | attempt={record.get('attempt')} | parse_ok={record.get('parse_ok')}")
        print("-" * 88)
        print(shorten_text(record.get("raw", ""), char_limit))


def show_model_output_table(prefix: str | None = None, limit: int | None = None) -> None:
    records = model_calls(prefix)
    if limit is not None:
        records = records[-limit:]
    rows = [
        {
            "label": record.get("label"),
            "attempt": record.get("attempt"),
            "parse_ok": record.get("parse_ok"),
            "raw_preview": shorten_text(record.get("raw", ""), 320),
        }
        for record in records
    ]
    show(pd.DataFrame(rows) if pd is not None else rows)


def evidence_text(plan: dict) -> str:
    evidence = plan.get("behavior_evidence", [])
    if isinstance(evidence, list):
        return "; ".join(str(item) for item in evidence)
    return str(evidence)


def schedule_table(schedule: list[dict]):
    rows = [
        {
            "row": i,
            "axis": item["axis_card"]["axis"],
            "target": item["target_trait"],
            "contrast": item["contrast_trait"],
            "style": item["style_id"],
            "length": item["length_bucket"],
            "evidence_strength": item.get("evidence_strength"),
            "label_mode": item.get("label_mode"),
            "trait_word": item.get("trait_word_should_appear"),
            "agency_subfacet": item.get("agency_subfacet", ""),
        }
        for i, item in enumerate(schedule)
    ]
    return pd.DataFrame(rows) if pd is not None else rows


def plans_table(plans: list[dict]):
    rows = [
        {
            "scenario_id": plan.get("scenario_id"),
            "axis": plan.get("axis"),
            "target": plan.get("target_property"),
            "contrast": plan.get("contrast_property"),
            "agent": plan.get("agent"),
            "setting": plan.get("setting"),
            "evidence": evidence_text(plan),
            "inference": plan.get("social_inference"),
            "label_mode": plan.get("label_mode"),
            "trait_word": plan.get("trait_word_should_appear"),
            "evidence_strength": plan.get("evidence_strength"),
            "agency_subfacet": plan.get("agency_subfacet", ""),
            "style": plan.get("style_id"),
            "length": plan.get("length_bucket"),
        }
        for plan in plans
    ]
    return pd.DataFrame(rows) if pd is not None else rows


def drafts_table(drafts: list[dict]):
    rows = []
    for draft in drafts:
        metadata = draft.get("metadata", {})
        plan = draft.get("scenario_plan", {})
        rows.append(
            {
                "scenario_id": draft.get("scenario_id"),
                "axis": metadata.get("axis"),
                "target": metadata.get("target_property"),
                "style": metadata.get("style_id"),
                "length": metadata.get("length_bucket"),
                "label_mode": metadata.get("label_mode"),
                "exact_trait": metadata.get("contains_exact_trait_word"),
                "evidence_strength": metadata.get("evidence_strength"),
                "agency_subfacet": metadata.get("agency_subfacet", ""),
                "evidence": evidence_text(plan),
                "text": draft.get("text"),
            }
        )
    return pd.DataFrame(rows) if pd is not None else rows


def checked_table(checked_rows: list[dict], drafts: list[dict]):
    drafts_by_id = {row.get("scenario_id"): row for row in drafts}
    rows = []
    for row in checked_rows:
        draft = drafts_by_id.get(row.get("scenario_id"), {})
        metadata = row.get("metadata", {})
        rows.append(
            {
                "scenario_id": row.get("scenario_id"),
                "target": metadata.get("target_property"),
                "label_mode": metadata.get("label_mode"),
                "verdict": row.get("verdict"),
                "reason": row.get("reason"),
                "draft_text": draft.get("text"),
                "checked_text": row.get("text"),
            }
        )
    return pd.DataFrame(rows) if pd is not None else rows


print("Inspection helpers are ready.")


### Step 1: Build The 5-Example Schedule

This is the target mix before any model calls. Inspect this first if you want to change axes, traits, styles, or lengths.


In [ ]:
def build_generation_schedule(n: int) -> list[dict]:
    schedule = []
    axis_cards = SOCIAL_ONTOLOGY
    agency_dominant_count = 0
    for i in range(n):
        axis_card = axis_cards[i % len(axis_cards)]
        traits = axis_card["anchor_traits"]
        target_trait = traits[(i // len(axis_cards)) % len(traits)]
        style = DISCOURSE_STYLES[(i * 3) % len(DISCOURSE_STYLES)]
        length_bucket = list(LENGTH_BUCKETS)[i % len(LENGTH_BUCKETS)]
        evidence_strength = EVIDENCE_STRENGTHS[(i // 2) % len(EVIDENCE_STRENGTHS)]
        label_mode = LABEL_MODE_SEQUENCE[i % len(LABEL_MODE_SEQUENCE)]
        trait_word_should_appear = label_mode != "implicit"
        agency_subfacet = ""
        if axis_card["axis"] == "agency_status":
            if target_trait == "dominant":
                agency_subfacet = ["prosocial_leadership", "controlling_dominance"][agency_dominant_count % 2]
                agency_dominant_count += 1
            else:
                agency_subfacet = choose_agency_subfacet(target_trait)
        schedule.append(
            {
                "axis_card": axis_card,
                "target_trait": target_trait,
                "contrast_trait": TRAIT_CONTRASTS.get(target_trait, ""),
                "style_id": style["style_id"],
                "length_bucket": length_bucket,
                "evidence_strength": evidence_strength,
                "label_mode": label_mode,
                "trait_word_should_appear": trait_word_should_appear,
                "agency_subfacet": agency_subfacet,
            }
        )
    random.shuffle(schedule)
    return schedule


schedule = build_generation_schedule(N_PILOT_EXAMPLES)
print(f"Scheduled {len(schedule)} examples. No model calls yet.")
show(schedule_table(schedule))


### Step 2: Generate Scenario Plans

This cell calls the model for plans only. It prints the raw model outputs before any export happens, then shows the parsed plans.


In [ ]:
RESET_MODEL_CALL_LOG = True

if RESET_MODEL_CALL_LOG:
    MODEL_CALL_LOG.clear()
    RAW_OUTPUT_LOG.clear()

scenario_plans = []

print(f"Generating {len(schedule)} scenario plans...")
for item in schedule:
    plan = make_social_property_plan(
        item["axis_card"],
        item["target_trait"],
        item["contrast_trait"],
        item["style_id"],
        item["length_bucket"],
        evidence_strength=item.get("evidence_strength"),
        label_mode=item.get("label_mode"),
        trait_word_should_appear=item.get("trait_word_should_appear"),
        agency_subfacet=item.get("agency_subfacet"),
    )
    if plan is not None:
        scenario_plans.append(plan)

print(f"Parsed scenario plans: {len(scenario_plans)} / {len(schedule)}")
if SHOW_RAW_OUTPUTS:
    print_model_outputs("plan:", limit=len(schedule))
else:
    show_model_output_table("plan:", limit=len(schedule))

show(plans_table(scenario_plans))


### Step 3: Inspect Scenario Plans

Look here for weird evidence, identity cues, overconfident social inferences, or style/length mismatches before rendering prose.


In [ ]:
if not scenario_plans:
    raise RuntimeError("Run Step 2 before inspecting plans.")

plan_rows = []
for plan in scenario_plans:
    scenario_content = {
        "agent": plan.get("agent"),
        "setting": plan.get("setting"),
        "behavior_evidence": plan.get("behavior_evidence", []),
        "social_inference": plan.get("social_inference"),
        "caveat": plan.get("caveat"),
    }
    plan_text = json.dumps(scenario_content, ensure_ascii=False)
    suspicious = []
    if PROTECTED_ATTRIBUTE_RE.search(plan_text):
        suspicious.append("protected_attribute_keyword")
    if IDENTITY_INFERENCE_RE.search(plan_text):
        suspicious.append("identity_inference_pattern")
    if APPEARANCE_RE.search(plan_text):
        suspicious.append("appearance_keyword")
    plan_rows.append(
        {
            "scenario_id": plan.get("scenario_id"),
            "axis": plan.get("axis"),
            "target": plan.get("target_property"),
            "contrast": plan.get("contrast_property"),
            "evidence_strength": plan.get("evidence_strength"),
            "label_mode": plan.get("label_mode"),
            "trait_word": plan.get("trait_word_should_appear"),
            "agency_subfacet": plan.get("agency_subfacet", ""),
            "evidence": evidence_text(plan),
            "inference": plan.get("social_inference"),
            "caveat": plan.get("caveat"),
            "suspicious": ", ".join(suspicious),
        }
    )

show(pd.DataFrame(plan_rows) if pd is not None else plan_rows)


### Step 4: Render Prose Drafts

This cell turns approved-looking plans into prose drafts. It prints raw render outputs and then displays each draft beside its source evidence.


In [ ]:
if not scenario_plans:
    raise RuntimeError("Run Step 2 before rendering prose.")

prose_draft = []
print(f"Rendering {len(scenario_plans)} prose drafts...")
for plan in scenario_plans:
    style = STYLE_BY_ID.get(plan.get("style_id"), random.choice(DISCOURSE_STYLES))
    draft = render_social_prose(plan, style)
    if draft is not None:
        prose_draft.append(draft)

print(f"Parsed prose drafts: {len(prose_draft)} / {len(scenario_plans)}")
if SHOW_RAW_OUTPUTS:
    print_model_outputs("render:", limit=len(scenario_plans))
else:
    show_model_output_table("render:", limit=len(scenario_plans))

show(drafts_table(prose_draft))


### Step 5: Inspect Draft Prose

This is the main qualitative checkpoint: read the generated text alongside target trait, source evidence, exact-label status, and word count.


In [ ]:
if not prose_draft:
    raise RuntimeError("Run Step 4 before inspecting drafts.")

draft_rows = []
for draft in prose_draft:
    metadata = draft.get("metadata", {})
    plan = draft.get("scenario_plan", {})
    text = draft.get("text", "")
    draft_rows.append(
        {
            "scenario_id": draft.get("scenario_id"),
            "axis": metadata.get("axis"),
            "target": metadata.get("target_property"),
            "evidence_strength": metadata.get("evidence_strength"),
            "label_mode": metadata.get("label_mode"),
            "agency_subfacet": metadata.get("agency_subfacet", ""),
            "trait_requested": metadata.get("trait_word_should_appear"),
            "exact_trait": metadata.get("contains_exact_trait_word"),
            "word_count": len(words(text)),
            "evidence": evidence_text(plan),
            "text": text,
        }
    )

show(pd.DataFrame(draft_rows) if pd is not None else draft_rows)


### Step 6: Run Edit / Check

This calls the editor only after you have inspected drafts. It displays raw editor outputs and a draft-vs-checked comparison.


In [ ]:
if not prose_draft:
    raise RuntimeError("Run Step 4 before edit/check.")

prose_checked = []
print(f"Checking {len(prose_draft)} drafts...")
for draft in prose_draft:
    prose_checked.append(edit_or_check_social_prose(draft))

print(f"Checked examples: {len(prose_checked)}")
if SHOW_RAW_OUTPUTS:
    print_model_outputs("edit:", limit=len(prose_draft))
else:
    show_model_output_table("edit:", limit=len(prose_draft))

show(checked_table(prose_checked, prose_draft))


### Step 7: Inspect Checked Examples

Use this to see what the editor kept, revised, or rejected before the programmatic audit.


In [ ]:
if not prose_checked:
    raise RuntimeError("Run Step 6 before inspecting checked examples.")

checked_rows = []
for row in prose_checked:
    metadata = row.get("metadata", {})
    checked_rows.append(
        {
            "scenario_id": row.get("scenario_id"),
            "axis": metadata.get("axis"),
            "target": metadata.get("target_property"),
            "evidence_strength": metadata.get("evidence_strength"),
            "label_mode": metadata.get("label_mode"),
            "agency_subfacet": metadata.get("agency_subfacet", ""),
            "verdict": row.get("verdict"),
            "reason": row.get("reason"),
            "word_count": len(words(row.get("text", ""))),
            "text": row.get("text"),
        }
    )

show(pd.DataFrame(checked_rows) if pd is not None else checked_rows)


### Step 8: Run Programmatic Audit

This attaches warnings and hard-drop statuses in memory only. It does not write files.


In [ ]:
if not prose_checked:
    raise RuntimeError("Run Step 6 before auditing.")

audit_input = [row for row in prose_checked if row.get("verdict") != "reject" and row.get("text")]
audited_examples, audit_report = audit_social_examples(audit_input)
final_rows = [row for row in audited_examples if row.get("audit_status") != "drop" and row.get("text")]
train_text = "\n\n".join(row["text"].strip() for row in final_rows)

print(json.dumps({k: audit_report[k] for k in ["total_examples", "kept_examples", "warned_examples", "dropped_examples"]}, indent=2))
print(f"Approx whitespace tokens in in-memory train_text: {len(train_text.split())}")
print("No files written in this cell.")

show(as_table(audit_report["audit_flag_counts"], "flag"))
show(as_table(audit_report["axis_distribution"], "axis"))
show(as_table(audit_report["target_trait_distribution"], "target_trait"))
show(as_table(audit_report["evidence_strength_distribution"], "evidence_strength"))
show(as_table(audit_report["label_mode_distribution"], "label_mode"))
show(as_table(audit_report["agency_subfacet_balance"], "agency_subfacet"))
show(as_table(audit_report["exact_trait_label_density_by_axis"], "axis", "exact_label_density"))
show(as_table(audit_report["exact_trait_label_density_by_target_trait"], "target_trait", "exact_label_density"))
show(as_table(audit_report["connective_rate_per_example"], "connective", "rate_per_example"))
print(f"Exact trait label density: {audit_report['exact_trait_label_density']:.3f}")
print(f"Single-moment 'is [trait]' rate: {audit_report['single_moment_is_trait_rate']:.3f}")

if audit_report.get("recommendations"):
    print("Recommendations:")
    for rec in audit_report["recommendations"]:
        print("-", rec)


## 9. Display Inspection

After the audit, inspect random final examples, warning rows, rejected rows, repeated openings, connective rates, distributions, exact trait label density, and name/pronoun balance.


In [ ]:
if "audited_examples" not in globals() or "audit_report" not in globals():
    raise RuntimeError("Run Step 8 before final inspection.")

final_examples = [row for row in audited_examples if row.get("audit_status") != "drop"]
warning_examples = [row for row in audited_examples if row.get("audit_status") == "warn"]
rejected_examples = [row for row in prose_checked if row.get("verdict") == "reject"]

rng = random.Random(RANDOM_SEED)
sample = rng.sample(final_examples, min(10, len(final_examples))) if final_examples else []

sample_rows = [
    {
        "scenario_id": row.get("scenario_id"),
        "axis": row.get("metadata", {}).get("axis"),
        "target": row.get("metadata", {}).get("target_property"),
        "evidence_strength": row.get("metadata", {}).get("evidence_strength"),
        "label_mode": row.get("metadata", {}).get("label_mode"),
        "agency_subfacet": row.get("metadata", {}).get("agency_subfacet", ""),
        "style": row.get("metadata", {}).get("style_id"),
        "audit_status": row.get("audit_status"),
        "text": row.get("text"),
    }
    for row in sample
]
show(pd.DataFrame(sample_rows) if pd is not None else sample_rows)

warning_rows = [
    {
        "scenario_id": row.get("scenario_id"),
        "target": row.get("metadata", {}).get("target_property"),
        "evidence_strength": row.get("metadata", {}).get("evidence_strength"),
        "label_mode": row.get("metadata", {}).get("label_mode"),
        "flags": ", ".join(flag["code"] for flag in row.get("audit_flags", [])),
        "text": row.get("text"),
    }
    for row in warning_examples[:20]
]
show(pd.DataFrame(warning_rows) if pd is not None else warning_rows)

rejected_rows = [
    {
        "scenario_id": row.get("scenario_id"),
        "reason": row.get("reason") or row.get("metadata", {}).get("editor_reason"),
        "text": row.get("text"),
    }
    for row in rejected_examples
]
show(pd.DataFrame(rejected_rows) if pd is not None else rejected_rows)

show(as_table(dict(audit_report["top_repeated_opening_5"]), "first_5_words"))
show(as_table(audit_report["style_distribution"], "style_id"))
show(as_table(audit_report["length_distribution"], "length_bucket"))
show(as_table(audit_report["axis_distribution"], "axis"))
show(as_table(audit_report["target_trait_distribution"], "target_trait"))
show(as_table(audit_report["evidence_strength_distribution"], "evidence_strength"))
show(as_table(audit_report["label_mode_distribution"], "label_mode"))
show(as_table(audit_report["agency_subfacet_balance"], "agency_subfacet"))
show(as_table(audit_report["pronoun_counts"], "pronoun"))
show(as_table(audit_report["possible_name_counts"], "possible_name"))

print(f"Exact trait label density: {audit_report['exact_trait_label_density']:.3f}")


## 10. Export Pilot Files

Run this only after inspection. By default, this cell skips writing. Set `EXPORT_NOW = True` in the cell when the in-memory examples look good.


In [ ]:
EXPORT_NOW = WRITE_FILES

if not EXPORT_NOW:
    print("Export skipped. Set EXPORT_NOW = True in this cell, or set SOCIAL_PROPERTIES_WRITE_FILES=1 before launch, then rerun this cell.")
else:
    required = ["scenario_plans", "prose_draft", "prose_checked", "audited_examples", "audit_report", "train_text"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(f"Run the staged generation/audit cells first. Missing: {missing}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_jsonl(OUTPUT_DIR / "raw_model_outputs.jsonl", MODEL_CALL_LOG)
    write_jsonl(OUTPUT_DIR / "scenario_plans.jsonl", scenario_plans)
    write_jsonl(OUTPUT_DIR / "prose_draft.jsonl", prose_draft)
    write_jsonl(OUTPUT_DIR / "prose_checked.jsonl", prose_checked)
    write_jsonl(OUTPUT_DIR / "examples_audited.jsonl", audited_examples)
    write_json(OUTPUT_DIR / "audit_report.json", audit_report)
    (OUTPUT_DIR / "train_pilot.txt").write_text(train_text.strip() + "\n", encoding="utf-8")

    print(f"Wrote pilot files to {OUTPUT_DIR}")
    print(f"Final kept/warn examples: {len(final_rows)}")
    print(f"Approx whitespace tokens in train_pilot.txt: {len(train_text.split())}")


## 11. Release GPU Memory

Run this cell when you are done with local-model testing. Closing the browser tab often leaves the Jupyter kernel alive, so this explicit cleanup is the most reliable way to release model memory without shutting down the kernel manually.


In [ ]:
release_model_memory()


## 12. Keep It Small

When the pilot looks safe and varied, the next scaling step is a separate 100k-token pilot with more careful balancing, batch-level prompting, and stronger audit/edit loops. This scratch notebook is only for the tiny local-model test.
